# Create Delta table and test Primary constraint

- The source comes from jupyter-pyspark/f1-sourcefiles
- Create delta lake table with primary key
- Import circuits.csv file into dataframe
- Add an extra row to break primary key contraint
- Insert data into delta table


# Initalise a spark session

In [1]:
# Initalise a spark session
import os
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType,
    DoubleType, DateType, BooleanType
)
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Fix JAVA_HOME to your actual Java 21 path
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages io.delta:delta-spark_2.12:3.2.0 pyspark-shell"

# Build Spark session with Delta Lake support
builder = SparkSession.builder \
    .appName("DeltaLakeExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()


26/06/16 22:10:39 WARN Utils: Your hostname, DESKTOP-3T7H98Q resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/16 22:10:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/ryip/projects/pyspark-deltalake/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ryip/.ivy2/cache
The jars for the packages stored in: /home/ryip/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a297be6f-9be6-489b-9af1-df0ffd574a49;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 237ms :: artifacts dl 7ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   | 

# Setup schema (database)


In [2]:
spark.conf.get("spark.sql.warehouse.dir")

'file:/home/ryip/projects/pyspark-deltalake/jupyter-pyspark/spark-warehouse'

In [3]:
spark.sql("CREATE DATABASE IF NOT EXISTS f1 COMMENT 'f1 schema'")
spark.sql("USE f1")



# Drop table if exists


In [4]:
spark.sql("DROP TABLE IF EXISTS f1.circuits")

spark.sql("DROP TABLE IF EXISTS f1.races")


DataFrame[]

# Create circuits table in f1 schema from csv files

In [8]:
# 1. Read the CSV file into a DataFrame
df_circuits = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("f1-sourcefiles/circuits.csv")

# 2. Save the DataFrame as a managed table 
df_circuits.write.saveAsTable("f1.circuits")




# Read the circuits table defintion from the Spark Catelog 

In [12]:
spark.sql(f"""
   -- Returns column names, data types, and any comments
    DESCRIBE f1.circuits;
""").show()

spark.sql(f"""
    -- Returns column details PLUS physical location, partition info, and table properties
    DESCRIBE EXTENDED f1.circuits;
""").show()

+----------+---------+-------+
|  col_name|data_type|comment|
+----------+---------+-------+
| circuitId|      int|   NULL|
|circuitRef|   string|   NULL|
|      name|   string|   NULL|
|  location|   string|   NULL|
|   country|   string|   NULL|
|       lat|   double|   NULL|
|       lng|   double|   NULL|
|       alt|      int|   NULL|
|       url|   string|   NULL|
+----------+---------+-------+

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|           circuitId|                 int|   NULL|
|          circuitRef|              string|   NULL|
|                name|              string|   NULL|
|            location|              string|   NULL|
|             country|              string|   NULL|
|                 lat|              double|   NULL|
|                 lng|              double|   NULL|
|                 alt|                 int|   NULL|
|                 url|  

# The show the dataframe definition

In [29]:
df_circuits.printSchema()

df_circuits.show()

root
 |-- circuitId: integer (nullable = true)
 |-- circuitRef: string (nullable = true)
 |-- name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- country: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- lng: double (nullable = true)
 |-- alt: integer (nullable = true)
 |-- url: string (nullable = true)

+---------+--------------+--------------------+------------+---------+--------+---------+---+--------------------+
|circuitId|    circuitRef|                name|    location|  country|     lat|      lng|alt|                 url|
+---------+--------------+--------------------+------------+---------+--------+---------+---+--------------------+
|        1|   albert_park|Albert Park Grand...|   Melbourne|Australia|-37.8497|  144.968| 10|http://en.wikiped...|
|        2|        sepang|Sepang Internatio...|Kuala Lumpur| Malaysia| 2.76083|  101.738| 18|http://en.wikiped...|
|        3|       bahrain|Bahrain Internati...|      Sakhir|  Bahrain| 26.0325|

# Create races table in f1 schema from csv files


In [9]:
# 1. Read the CSV file into a DataFrame
df_races = spark.read.csv("f1-sourcefiles/races.csv", header=True, inferSchema=True)

# 2. Save the DataFrame as a managed table
df_races.write.saveAsTable("f1.races") 


# Read the races table defintion from the Spark Catalog 

In [16]:
spark.sql(f"""
   -- Returns column names, data types, and any comments
    DESCRIBE f1.races;
""").show(1000)

spark.sql(f"""
    -- Returns column details PLUS physical location, partition info, and table properties
    DESCRIBE EXTENDED f1.races;
""").show(1000)

+-----------+---------+-------+
|   col_name|data_type|comment|
+-----------+---------+-------+
|     raceId|      int|   NULL|
|       year|      int|   NULL|
|      round|      int|   NULL|
|  circuitId|      int|   NULL|
|       name|   string|   NULL|
|       date|     date|   NULL|
|       time|   string|   NULL|
|        url|   string|   NULL|
|   fp1_date|   string|   NULL|
|   fp1_time|   string|   NULL|
|   fp2_date|   string|   NULL|
|   fp2_time|   string|   NULL|
|   fp3_date|   string|   NULL|
|   fp3_time|   string|   NULL|
| quali_date|   string|   NULL|
| quali_time|   string|   NULL|
|sprint_date|   string|   NULL|
|sprint_time|   string|   NULL|
+-----------+---------+-------+

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|              raceId|                 int|   NULL|
|                year|                 int|   NULL|
|               round|             

# The races dataframe definition

In [28]:
df_races.printSchema()

df_races.show()

+------+----+-----+---------+--------------------+----------+--------+--------------------+--------+--------+--------+--------+--------+--------+----------+----------+-----------+-----------+
|raceId|year|round|circuitId|                name|      date|    time|                 url|fp1_date|fp1_time|fp2_date|fp2_time|fp3_date|fp3_time|quali_date|quali_time|sprint_date|sprint_time|
+------+----+-----+---------+--------------------+----------+--------+--------------------+--------+--------+--------+--------+--------+--------+----------+----------+-----------+-----------+
|     1|2009|    1|        1|Australian Grand ...|2009-03-29|06:00:00|http://en.wikiped...|      \N|      \N|      \N|      \N|      \N|      \N|        \N|        \N|         \N|         \N|
|     2|2009|    2|        2|Malaysian Grand Prix|2009-04-05|09:00:00|http://en.wikiped...|      \N|      \N|      \N|      \N|      \N|      \N|        \N|        \N|         \N|         \N|
|     3|2009|    3|       17|  Chinese G

# INNER JOIN 2 TABLES - No variables dataframes

In [25]:
# spark.sql("SELECT r.raceid, r.year, r.round, r.name, r.date, r.time, r.circuitId, c.name as circuit_name FROM f1.circuits c INNER JOIN f1.races r ON c.circuitId = r.circuitId LIMIT 20").show()

spark.sql("SELECT r.*, c.* FROM f1.circuits c INNER JOIN f1.races r ON c.circuitId = r.circuitId LIMIT 20").show()

26/06/16 23:58:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------+----+-----+---------+--------------------+----------+--------+--------------------+--------+--------+--------+--------+--------+--------+----------+----------+-----------+-----------+---------+-----------+--------------------+------------+---------+--------+---------+---+--------------------+
|raceId|year|round|circuitId|                name|      date|    time|                 url|fp1_date|fp1_time|fp2_date|fp2_time|fp3_date|fp3_time|quali_date|quali_time|sprint_date|sprint_time|circuitId| circuitRef|                name|    location|  country|     lat|      lng|alt|                 url|
+------+----+-----+---------+--------------------+----------+--------+--------------------+--------+--------+--------+--------+--------+--------+----------+----------+-----------+-----------+---------+-----------+--------------------+------------+---------+--------+---------+---+--------------------+
|     1|2009|    1|        1|Australian Grand ...|2009-03-29|06:00:00|http://en.wikiped...|   

# Dataframe inner join

In [30]:
result_df = df_circuits.join(
    df_races, 
    df_circuits.circuitId == df_races.circuitId, 
    "inner"
)

result_df.show()


+---------+-----------+--------------------+------------+---------+--------+---------+---+--------------------+------+----+-----+---------+--------------------+----------+--------+--------------------+--------+--------+--------+--------+--------+--------+----------+----------+-----------+-----------+
|circuitId| circuitRef|                name|    location|  country|     lat|      lng|alt|                 url|raceId|year|round|circuitId|                name|      date|    time|                 url|fp1_date|fp1_time|fp2_date|fp2_time|fp3_date|fp3_time|quali_date|quali_time|sprint_date|sprint_time|
+---------+-----------+--------------------+------------+---------+--------+---------+---+--------------------+------+----+-----+---------+--------------------+----------+--------+--------------------+--------+--------+--------+--------+--------+--------+----------+----------+-----------+-----------+
|        1|albert_park|Albert Park Grand...|   Melbourne|Australia|-37.8497|  144.968| 10|http

# Update circuits table

In [35]:
# Original circuitRef albert_park
spark.sql(f"""
    UPDATE f1.circuits 
    SET circuitRef = 'new_albert_park' 
    WHERE circuitId = 1;
""")

DataFrame[num_affected_rows: bigint]

# Check update of circuits table

In [38]:
spark.sql(f"""
    SELECT c.* FROM f1.circuits c 
    WHERE circuitId = 1;
""").show()

+---------+---------------+--------------------+---------+---------+--------+-------+---+--------------------+
|circuitId|     circuitRef|                name| location|  country|     lat|    lng|alt|                 url|
+---------+---------------+--------------------+---------+---------+--------+-------+---+--------------------+
|        1|new_albert_park|Albert Park Grand...|Melbourne|Australia|-37.8497|144.968| 10|http://en.wikiped...|
+---------+---------------+--------------------+---------+---------+--------+-------+---+--------------------+



# Convert to Delta table

In [33]:
spark.sql(f"""
    CONVERT TO DELTA f1.circuits;
""")

DataFrame[]

# Check table is now delta table

In [34]:
spark.sql(f"""
    -- Returns column details PLUS physical location, partition info, and table properties
    DESCRIBE EXTENDED f1.circuits;
""").show(1000)

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|           circuitId|                 int|   NULL|
|          circuitRef|              string|   NULL|
|                name|              string|   NULL|
|            location|              string|   NULL|
|             country|              string|   NULL|
|                 lat|              double|   NULL|
|                 lng|              double|   NULL|
|                 alt|                 int|   NULL|
|                 url|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|                Name|spark_catalog.f1....|       |
|                Type|             MANAGED|       |
|            Location|file:/home/ryip/p...|       |
|            Provider|               delta|       |
|    Table Properties|[delta.minReaderV...|       |
+-----------

# Show the constraint is added

In [39]:
# Show table properties including constraints
spark.sql("SHOW TBLPROPERTIES f1.circuits").show(truncate=False)

+------------------------------------+---------------------+
|key                                 |value                |
+------------------------------------+---------------------+
|delta.autoOptimize.autoCompact      |true                 |
|delta.autoOptimize.optimizeWrite    |true                 |
|delta.constraints.circuitid_not_null|circuitId IS NOT NULL|
|delta.minReaderVersion              |1                    |
|delta.minWriterVersion              |3                    |
+------------------------------------+---------------------+



# Read the csv into a dataframe

In [40]:

df = spark.read.csv("f1-sourcefiles/circuits.csv", header=True, inferSchema=True)

df.show()

df.printSchema()


+---------+--------------+--------------------+------------+---------+--------+---------+---+--------------------+
|circuitId|    circuitRef|                name|    location|  country|     lat|      lng|alt|                 url|
+---------+--------------+--------------------+------------+---------+--------+---------+---+--------------------+
|        1|   albert_park|Albert Park Grand...|   Melbourne|Australia|-37.8497|  144.968| 10|http://en.wikiped...|
|        2|        sepang|Sepang Internatio...|Kuala Lumpur| Malaysia| 2.76083|  101.738| 18|http://en.wikiped...|
|        3|       bahrain|Bahrain Internati...|      Sakhir|  Bahrain| 26.0325|  50.5106|  7|http://en.wikiped...|
|        4|     catalunya|Circuit de Barcel...|    Montmeló|    Spain|   41.57|  2.26111|109|http://en.wikiped...|
|        5|      istanbul|       Istanbul Park|    Istanbul|   Turkey| 40.9517|   29.405|130|http://en.wikiped...|
|        6|        monaco|   Circuit de Monaco| Monte-Carlo|   Monaco| 43.7347| 

# Add a dummy row with a NOT NULL circuitId

In [41]:
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# Use the same schema as your existing df
new_row = spark.createDataFrame([
    Row(circuitId=None, circuitRef="London", name="Test Circuit", location="London", country="UK", lat=51.5, lng=-0.1, alt=10, url="http://test.com")
], schema=df.schema)

df = df.unionByName(new_row)

df.show(df.count(), truncate=False)

+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|circuitId|circuitRef    |name                                 |location             |country      |lat     |lng      |alt |url                                                                    |
+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|1        |albert_park   |Albert Park Grand Prix Circuit       |Melbourne            |Australia    |-37.8497|144.968  |10  |http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit              |
|2        |sepang        |Sepang International Circuit         |Kuala Lumpur         |Malaysia     |2.76083 |101.738  |18  |http://en.wikipedia.org/wiki/Sepang_International_Circuit              |
|3        |bahr

# This code removes the last row if I have added extra row by accident

In [44]:
df = df.limit(df.count() - 1)

df.show(df.count(), truncate=False)

+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|circuitId|circuitRef    |name                                 |location             |country      |lat     |lng      |alt |url                                                                    |
+---------+--------------+-------------------------------------+---------------------+-------------+--------+---------+----+-----------------------------------------------------------------------+
|1        |albert_park   |Albert Park Grand Prix Circuit       |Melbourne            |Australia    |-37.8497|144.968  |10  |http://en.wikipedia.org/wiki/Melbourne_Grand_Prix_Circuit              |
|2        |sepang        |Sepang International Circuit         |Kuala Lumpur         |Malaysia     |2.76083 |101.738  |18  |http://en.wikipedia.org/wiki/Sepang_International_Circuit              |
|3        |bahr

# Overwrite the data in the f1.circuits table

With a null circuitId row the insert will fail
Then remove the row with a null primary key - use the code above
And run the insert again - this should succeed


In [45]:
df.write.format("delta").mode("overwrite").saveAsTable("f1.circuits")

In [ ]:
# SELECT from the Delta tabke f1.circuits

In [48]:
spark.sql(f"""
   SELECT * FROM f1.circuits WHERE country = 'UK'
""").show()

+---------+------------+-------------------+----------------+-------+-------+--------+---+--------------------+
|circuitId|  circuitRef|               name|        location|country|    lat|     lng|alt|                 url|
+---------+------------+-------------------+----------------+-------+-------+--------+---+--------------------+
|        9| silverstone|Silverstone Circuit|     Silverstone|     UK|52.0786|-1.01694|153|http://en.wikiped...|
|       31|   donington|     Donington Park|Castle Donington|     UK|52.8306|-1.37528| 88|http://en.wikiped...|
|       38|brands_hatch|       Brands Hatch|            Kent|     UK|51.3569|0.263056|145|http://en.wikiped...|
|       58|     aintree|            Aintree|       Liverpool|     UK|53.4769|-2.94056| 20|http://en.wikiped...|
+---------+------------+-------------------+----------------+-------+-------+--------+---+--------------------+

